In [2]:
import pandas as pd

# Load the cleaned dataset
df = pd.read_csv('OneDrive/Desktop/LAST Project/Results.csv',header=None)

# Assign column names
df.columns = [
    'Invoice', 'StockCode', 'Description',
    'Quantity', 'InvoiceDate', 'Price',
    'Customer_ID', 'Country', 'SourceSheet'
]

# Convert InvoiceDate into datetime format
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Check dataset dimensions
print(df.shape)

# Check column information
df.info()

C:\Users\Kashish\AppData\Local\Temp\ipykernel_27620\3431605650.py:4: DtypeWarning: Columns (0: 0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('OneDrive/Desktop/LAST Project/Results.csv',header=None)


(1033036, 9)
<class 'pandas.DataFrame'>
RangeIndex: 1033036 entries, 0 to 1033035
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1033036 non-null  object        
 1   StockCode    1033036 non-null  str           
 2   Description  1028761 non-null  str           
 3   Quantity     1033036 non-null  int64         
 4   InvoiceDate  1033036 non-null  datetime64[ns]
 5   Price        1033036 non-null  float64       
 6   Customer_ID  797885 non-null   float64       
 7   Country      1033036 non-null  str           
 8   SourceSheet  1033036 non-null  str           
dtypes: datetime64[ns](1), float64(2), int64(1), object(1), str(4)
memory usage: 126.3+ MB


In [3]:
# FILTER DATA AND CALCULATE RFM

# Keep customers with valid Customer IDs and positive quantities
rfm_data = df[
    (df['Customer_ID'].notna()) &
    (df['Quantity'] > 0)
].copy()

# Set snapshot date
snapshot_date = pd.to_datetime('2011-12-10')

# Calculate Recency, Frequency and Monetary
rfm = rfm_data.groupby('Customer_ID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('Quantity', lambda x: (
        x * rfm_data.loc[x.index, 'Price']
    ).sum())
).reset_index()

# Display first five customers
rfm.head()

,Customer_ID,Recency,Frequency,Monetary
0,12346.0,325,12,77556.46
1,12347.0,2,8,4921.53
2,12348.0,75,5,2019.40
3,12349.0,18,4,4428.69
4,12350.0,310,1,334.40


In [4]:
# ASSIGN RFM SCORES

# Recency: Lower days = better score
rfm['R_Score'] = pd.qcut(
    rfm['Recency'],
    5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

# Frequency: Higher invoice count = better score
rfm['F_Score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

# Monetary: Higher spending = better score
rfm['M_Score'] = pd.qcut(
    rfm['Monetary'],
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

# Display scores
rfm.head()

,Customer_ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score
0,12346.0,325,12,77556.46,2,5,5
1,12347.0,2,8,4921.53,5,4,5
2,12348.0,75,5,2019.40,3,4,4
3,12349.0,18,4,4428.69,5,3,5
4,12350.0,310,1,334.40,2,1,2


In [5]:
# ASSIGN CUSTOMER SEGMENTS

def segment_customer(row):

    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champions'

    elif row['R_Score'] >= 3 and row['F_Score'] >= 3:
        return 'Loyal'

    elif row['R_Score'] <= 2 and row['F_Score'] >= 3:
        return 'At Risk'

    elif row['R_Score'] >= 4 and row['F_Score'] <= 2:
        return 'New'

    elif row['R_Score'] == 3 and row['F_Score'] <= 2:
        return 'Needs Attention'

    else:
        return 'Lost'


rfm['Segment'] = rfm.apply(segment_customer, axis=1)

# Identify customers inactive for more than 90 days
rfm['IsChurned'] = rfm['Recency'] > 90

# Display results
rfm.head()

,Customer_ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,Segment,IsChurned
0,12346.0,325,12,77556.46,2,5,5,At Risk,True
1,12347.0,2,8,4921.53,5,4,5,Champions,False
2,12348.0,75,5,2019.40,3,4,4,Loyal,False
3,12349.0,18,4,4428.69,5,3,5,Loyal,False
4,12350.0,310,1,334.40,2,1,2,Lost,True


In [6]:
#  MERGE RFM RESULTS AND EXPORT

# Prepare transaction data
df = df[
    (df['Customer_ID'].notna()) &
    (df['Quantity'] > 0)
].copy()

# Create Invoice Month
df['InvoiceMonth'] = (
    df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()
)

# Calculate transaction revenue
df['Revenue'] = df['Quantity'] * df['Price']

# Merge customer-level RFM results with transactions
df_final = df.merge(
    rfm[
        ['Customer_ID', 'Recency', 'Frequency', 'Monetary',
         'R_Score', 'F_Score', 'M_Score', 'Segment', 'IsChurned']
    ],
    on='Customer_ID',
    how='left'
)

# Export final dataset to Excel
df_final.to_excel('EcommerceDashboardData.xlsx', index=False)

print("Final dataset exported successfully!")
print("Total transaction records:", len(df_final))
print("Total customers:", df_final['Customer_ID'].nunique())

Final dataset exported successfully!
Total transaction records: 779495
Total customers: 5881
